## Create boto3 session and client

In [7]:
import boto3
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)

aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_REGION")
aws_account_id = os.getenv("AWS_ACCOUNT_ID")
DATA_AUTOMATION_PROFILE_ARN = (
    f'arn:aws:bedrock:{aws_region}:{aws_account_id}:data-automation-profile/{os.getenv("BDA_PROFILE_ID")}'
)
BDA_PROJECT_ARN = os.getenv("BDA_PROJECT_ARN")

sesion_aws = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

bedrock_bda_client = sesion_aws.client('bedrock-data-automation')
bedrock_bda_runtime_client = sesion_aws.client('bedrock-data-automation-runtime')
s3_client = sesion_aws.client('s3')

## Invoke BDA 1 Document

In [4]:
# Rutas de ejemplo
INPUT_S3_URI = "s3://test-source-s3-vector/cars/ford-centroamerica-bronco-2022-catalogo-descargable-esp.pdf"
OUTPUT_S3_URI = "s3://test-source-s3-vector/bda-output-parser/"


response = bedrock_bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': INPUT_S3_URI
    },
    outputConfiguration={
        's3Uri': OUTPUT_S3_URI
    },
    dataAutomationConfiguration={
        # Si usaste un proyecto:
        'dataAutomationProjectArn': BDA_PROJECT_ARN,
        'stage': 'LIVE'
    },
    dataAutomationProfileArn=DATA_AUTOMATION_PROFILE_ARN
)
print(f"ARN de Invocación BDA: {response['invocationArn']}")
# Debes monitorear este ARN para verificar la finalización del trabajo.

ARN de Invocación BDA: arn:aws:bedrock:us-east-1:376710739935:data-automation-invocation/9dff4225-0dd3-43e4-8db6-a68acf139265


## GET Status

In [8]:
response = bedrock_bda_runtime_client.get_data_automation_status(
    invocationArn='arn:aws:bedrock:us-east-1:376710739935:data-automation-invocation/9dff4225-0dd3-43e4-8db6-a68acf139265'
)
response

{'status': 'Success',
 'outputConfiguration': {'s3Uri': 's3://test-source-s3-vector/bda-output-parser//9dff4225-0dd3-43e4-8db6-a68acf139265/job_metadata.json'},
 'ResponseMetadata': {'RequestId': 'ee172b26-5910-43cf-a5e8-5480aa4adcd5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 27 Oct 2025 04:42:26 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '155',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'ee172b26-5910-43cf-a5e8-5480aa4adcd5'},
  'RetryAttempts': 0}}

## Pipeline to Process Documents with BDA 
1. Get all documents from s3 uri
2. Invoke BDA for each document found in the S3 path.

In [10]:
from typing import List, Dict
import time

# --- 1. Funciones del Pipeline de Invocación ---

def get_document_s3_uris(bucket_name: str, prefix: str, file_extension: str = '.pdf') -> List[str]:
    """
    1. Obtiene la lista de URIs S3 de los documentos en el path especificado.
    """
    print(f"\n1. Buscando documentos en s3://{bucket_name}/{prefix}")
    s3_uris = []
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    
    for page in pages:
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith(file_extension.lower()) and key != prefix:
                s3_uris.append(f"s3://{bucket_name}/{key}")
    
    print(f"-> {len(s3_uris)} documentos encontrados.")
    return s3_uris

def invoke_bda_pipeline(
    input_bucket: str, 
    input_prefix: str, 
    output_uri: str,
    file_extension: str = '.pdf'
) -> List[str]:
    """
    2. Invoca BDA para cada documento encontrado y devuelve una lista de ARN de invocación.
    """
    document_uris = get_document_s3_uris(input_bucket, input_prefix, file_extension)
    invocation_arns = []
    
    if not document_uris:
        print("No hay documentos para procesar.")
        return []

    print("\n2. Invocando Bedrock Data Automation (BDA) para cada documento...")

    for uri in document_uris:
        print(f"   -> Iniciando BDA para: {uri}")
        try:
            response = bedrock_bda_runtime_client.invoke_data_automation_async(
                inputConfiguration={'s3Uri': uri},
                outputConfiguration={'s3Uri': output_uri},
                dataAutomationConfiguration={
                    'dataAutomationProjectArn': BDA_PROJECT_ARN,
                    'stage': 'LIVE'
                },
                dataAutomationProfileArn=DATA_AUTOMATION_PROFILE_ARN
            )
            invocation_arn = response['invocationArn']
            invocation_arns.append(invocation_arn)
            print(f"   -> ARN generado: {invocation_arn.split('/')[-1]}")
            
        except Exception as e:
            print(f"   !!! ERROR al invocar BDA para {uri}: {e}")

    return invocation_arns

# --- 2. Funciones de Observabilidad (Monitoreo) ---

def get_jobs_status(invocation_arns: List[str]) -> Dict[str, str]:
    """
    Revisa el estado actual de una lista de ARN de invocación de BDA.
    """
    if not invocation_arns:
        print("No hay ARN de invocación para monitorear.")
        return {}

    print(f"\n--- Monitoreo de {len(invocation_arns)} Trabajos BDA ---")
    status_summary = {}

    for arn in invocation_arns:
        job_id = arn.split('/')[-1]
        try:
            response = bedrock_bda_runtime_client.get_data_automation_status(
                invocationArn=arn
            )
            current_status = response.get('status', 'STATUS_UNKNOWN')
            status_summary[job_id] = current_status
            print(f"[{job_id}] -> Estado: {current_status}")
        except Exception as e:
            status_summary[job_id] = f"ERROR_FETCHING_STATUS: {e}"
            print(f"[{job_id}] -> ERROR: No se pudo obtener el estado.")
            
    return status_summary

In [ ]:
INPUT_BUCKET = "test-source-s3-vector"
INPUT_PREFIX = "cars/"

from datetime import datetime

fecha_actual = datetime.now().strftime("%Y-%m-%d")


# La URI de salida donde se guardarán los resultados del parser.
OUTPUT_S3_URI = f"s3://{INPUT_BUCKET}/bda-output-parser/"

# A. Ejecutar el Pipeline e Invocar los Jobs
all_arns = invoke_bda_pipeline(INPUT_BUCKET, INPUT_PREFIX, OUTPUT_S3_URI, file_extension='.pdf')

# B. Monitorear el Estado Inicial de los Jobs
if all_arns:
    print("\nInvocaciones BDA completadas. Iniciando monitoreo...")
    
    # Monitoreo simple: revisa el estado cada 30 segundos
    current_statuses = {}
    pending_jobs = list(all_arns)
    
    # Limita las revisiones para no exceder los límites de API
    max_checks = 10
    check_count = 0
    
    while pending_jobs and check_count < max_checks:
        print(f"\n--- REVISIÓN DE ESTADO ({check_count + 1}/{max_checks}) ---")
        
        # Revisar solo los trabajos pendientes
        pending_jobs_to_check = list(pending_jobs)
        status_update = get_jobs_status(pending_jobs_to_check)
        
        newly_completed_jobs = []
        for arn in pending_jobs:
            job_id = arn.split('/')[-1]
            status = status_update.get(job_id)
            
            # Si el trabajo terminó, lo quitamos de la lista de pendientes
            if status in ['COMPLETED', 'FAILED', 'ERROR']:
                newly_completed_jobs.append(arn)
        
        # Actualizar la lista de trabajos pendientes
        pending_jobs = [arn for arn in pending_jobs if arn not in newly_completed_jobs]
        
        if pending_jobs:
            print(f"Quedan {len(pending_jobs)} trabajos pendientes. Esperando 5 segundos...")
            time.sleep(5)
            check_count += 1

    print("\n\n--- MONITOREO FINALIZADO ---")
    final_status = get_jobs_status(all_arns)
    print("Resumen de estados finales:")
    print(json.dumps(final_status, indent=2))
else:
    print("\nNo se pudo invocar ningún trabajo BDA.")


1. Buscando documentos en s3://test-source-s3-vector/cars/
-> 8 documentos encontrados.

2. Invocando Bedrock Data Automation (BDA) para cada documento...
   -> Iniciando BDA para: s3://test-source-s3-vector/cars/ford-centroamerica-bronco-2022-catalogo-descargable-esp.pdf
   -> ARN generado: 42b40ab4-35b6-4d60-b5f3-df73e596a716
   -> Iniciando BDA para: s3://test-source-s3-vector/cars/ford-centroamerica-bronco-sport-2021-catalogo-descargable-esp.pdf
   -> ARN generado: 10ee40f3-6348-4d67-a74f-13c84d7ba6cc
   -> Iniciando BDA para: s3://test-source-s3-vector/cars/ford-centroamerica-edge-2022-catalogo-descargable-esp.pdf
   -> ARN generado: b003a6a6-b0de-4c54-bf42-0894c4cafe5c
   -> Iniciando BDA para: s3://test-source-s3-vector/cars/ford-centroamerica-escape-2022-catalogo-descargable-esp.pdf
   -> ARN generado: 937423c1-9b5a-4d19-8369-23110825583e
   -> Iniciando BDA para: s3://test-source-s3-vector/cars/ford-centroamerica-everest-2023-catalogo-descargable-esp.pdf
   -> ARN generado: c